In [ ]:
import pybamm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
import os, sys
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))
from batfuns import *
plt.rcParams = set_rc_params(plt.rcParams)

eSOH_DIR = "../data/esoh_R/"
oCV_DIR = "../data/ocv/"

In [ ]:
# get parameter values
parameter_values = get_parameter_values()
# define model
spm = pybamm.lithium_ion.SPM(
    {
        "SEI": "ec reaction limited",
        "loss of active material": "stress-driven",
        "lithium plating": "irreversible",
        "stress-induced diffusion": "false",
    }
)
#model internal parameters
param=spm.param

In [ ]:
cells = [3,9,12]
sno = 11
sim_des = f'cond{sno}'
irrev_exp_data=[]
del_sei=[]; del_li=[]; es_ic_n=[]; es_ic_p=[];
for cell in cells:
    # load the data for each cell
    # get parameters to intialize the model
    cell_no,dfe,dfe_0,dfo_0,N,N_0 = load_data(cell,eSOH_DIR,oCV_DIR)
    if cell == 3:
        Ns = np.insert(N_0[1:]-1,0,0)
        dfe = dfe_0
        Ncyc = N_0[-3]
        Ns = Ns[:-2]
    elif cell == 9:
        Ns = np.insert(N[1:]-1,0,0)
        Ncyc = N[-1]
    elif cell == 12:
        Ns = np.insert(N[1:]-1,0,0)
        Ncyc = N[-1]
    eps_n_data,eps_p_data,c_rate_c,c_rate_d,dis_set,Temp,SOC_0 = init_exp(cell_no,dfe,spm,parameter_values)
    experiment = pybamm.Experiment(
        [
            ("Discharge at "+c_rate_d+dis_set,
            "Rest for 10 sec",
            "Charge at "+c_rate_c+" until 4.2V", 
            "Hold at 4.2V until C/100")
        ] *Ncyc,
        termination="50% capacity",
    #     cccv_handling="ode",
    )
   
    par_val = {}
    # Room temp
    par_val[0] = [4.0312e-08,1.8157e-07,1.0776,2.3586e-09,-4.9170e-09,-1.4406e-09,4.60788219e-16,4.56607447e-19]
    # First Tuning
    par_val[1] = [5.6076e-08,6.7429e-07,1.02,1.4576e-08,-1.7447e-07,-2.5257e-08,4.60788219e-16,4.56607447e-19]
    # Contraint 1/2 and 2
    par_val[2] = [8.0624e-08,3.6314e-07,1.02,4.7172e-09,-9.8340e-09,-2.8812e-09,4.60788219e-16,4.56607447e-19]
    # Constraint 1/5 and 5
    par_val[3] = [9.8220e-08,9.0785e-07,1.02,1.1793e-08,-2.4585e-08,-7.2030e-09,4.60788219e-16,4.56607447e-19]
    # Constraint 1/5 and 5 with no constraints on beta2'
    par_val[4] = [6.2621e-08,6.8771e-07,1.02,1.1793e-08,-1.7385e-07,-2.4901e-08,4.60788219e-16,4.56607447e-19]
    # Constraint 1/2 and 2 with no constraints on beta2'
    par_val[5] = [8.0624e-08,3.6314e-07,1.002,4.7172e-09,-2.4217e-07,-2.5212e-08,4.60788219e-16,4.56607447e-19]
    # kpl=0, retune ksei
    par_val[6] = [1.2796e-07,7.2624e-07,1.02,0,-1.7764e-07,-1.4406e-11,4.9166e-17,4.56607447e-19]
    # kpl=0, retune ksei
    par_val[7] = [1.1375e-07,6.4051e-07,1.0,0,-1.5557e-07,-2.2287e-08,2.2675e-15,4.56607447e-19]
    # kpl=0, retune ksei, different cost function, better initial guess    
    par_val[8] = [9.1084e-08,5.9270e-07,1.00,0,-1.6967e-07,-3.0447e-08,1.5803e-15,4.56607447e-19]
    # kpl=0, retune ksei, cells 6 9 12
    par_val[9] = [2.8391e-07,2.0958e-06,1.9123,0,-8.6507e-07,-9.9184e-08,2.8281e-15,4.56607447e-19]
    # kpl=0, retune ksei, cells 3 9 12
    par_val[10] = [1.3292e-07,9.2142e-07,1.0,0,-1.2058e-07,-2.1779e-08,1.9308e-15,4.56607447e-19]
    # 2 step method with ksei
    par_val[11] = [1.1759e-07,8.9155e-07,1.0,7.5992e-09,-1.2611e-07,-2.3971e-08,1.4840e-15,4.56607447e-19]
    # 2 step method with dsei
    par_val[12] = [1.2057e-07,8.9269e-07,1.0,1.1355e-08,-1.3511e-07,-3.1561e-08,4.6079e-16,3.6445e-18]
    # 1 step method with dsei
    par_val[13] = [1.4653e-07,9.3369e-07,1.0,4.2122e-09,-1.3385e-07,-2.7939e-08,4.6079e-16,4.7510e-19]
    # First Tuning 3,9,12
    par_val[14] = [1.1944e-07,8.9283e-07,1.0,1.1646e-08,-1.3499e-07,-3.0872e-08,4.60788219e-16,4.56607447e-19]
    parameter_values = get_parameter_values()
    parameter_values.update(
        {
            "Negative electrode active material volume fraction": eps_n_data,
            "Positive electrode active material volume fraction": eps_p_data,
            "Initial temperature [K]": 273.15+Temp,
            "Ambient temperature [K]": 273.15+Temp,
            "Positive electrode LAM constant proportional term [s-1]": par_val[sno][0],
            "Negative electrode LAM constant proportional term [s-1]": par_val[sno][1],
            "Positive electrode LAM constant proportional term 2 [s-1]": par_val[sno][5],
            "Negative electrode LAM constant proportional term 2 [s-1]": par_val[sno][4],
            "Positive electrode LAM constant exponential term": par_val[sno][2],
            "Negative electrode LAM constant exponential term": par_val[sno][2],
            "SEI kinetic rate constant [m.s-1]":  par_val[sno][6], #1.08494281e-16 , 
            "EC diffusivity [m2.s-1]": par_val[sno][7],#8.30909086e-19,
            "SEI growth activation energy [J.mol-1]": 1.87422275e+04,#1.58777981e+04,
            "Lithium plating kinetic rate constant [m.s-1]": par_val[sno][3],
            "Initial inner SEI thickness [m]": 0e-09,
            "Initial outer SEI thickness [m]": 5e-09,
            "Li plating resistivity [Ohm.m]": 5*3e4,
            "SEI resistivity [Ohm.m]": 1.25*3e4,
            "Negative electrode partial molar volume [m3.mol-1]": 7e-06,
            "Negative electrode LAM min stress [Pa]": 0,
            "Negative electrode LAM max stress [Pa]": 0,
            "Positive electrode LAM min stress [Pa]": 0,
            "Positive electrode LAM max stress [Pa]": 0,
            "Negative electrode diffusion coefficient [m2.s-1]": 8e-14,
            "Positive electrode diffusion coefficient [m2.s-1]": 8e-15,
            # "Negative electrode critical stress [Pa]": 20e+06,
            # "Positive electrode critical stress [Pa]": 40e+06,
        },
        check_already_exists=False,
    )
    # simulate the model
    df = cycle_adaptive_simulation_V2(spm, parameter_values, experiment,SOC_0, save_at_cycles=1)
    # SEI layer thickness
    del_sei = np.append(del_sei,(df["X-averaged SEI thickness [m]"][Ns]-df["X-averaged SEI thickness [m]"][0]))
    # Plated Lithium thickness
    del_li = np.append(del_li,df["X-averaged lithium plating thickness [m]"][Ns])
    # negative electode inactive material
    es_ic_n = np.append(es_ic_n,(-df["X-averaged negative electrode active material volume fraction"][Ns]+df["X-averaged negative electrode active material volume fraction"][0]))
    # positive electrode inactive material 
    es_ic_p = np.append(es_ic_p,(-df["X-averaged positive electrode active material volume fraction"][Ns]+df["X-averaged positive electrode active material volume fraction"][0])) 
    irrev_exp_data = np.append(irrev_exp_data,dfe["irrev_exp"].to_numpy()[:len(Ns)])

In [ ]:
len(del_sei)

In [ ]:
len(irrev_exp_data)

In [ ]:
# cost function for tuning resistance
def fitfunc3(X,b1, b2, b3, b4):
  ln = 6.2e-05
  lp = 6.7e-05
  del_sei,del_li2,es_ic_n,es_ic_p=X
  # equation 57 in paper
  out = (b1*del_sei*1e6+b2*del_li2*1e12+b3*es_ic_n+b4*es_ic_p)
  return out

In [ ]:
# initial guesss
ig = (100,1000,1000,1000)
# lower bound
lb = 0
# upper bound
ub = [1e5,1e10,1e15,1e15]
# tune expansion coefficients
popt1, pcov1 = curve_fit(fitfunc3, (del_sei,del_li**2,es_ic_n,es_ic_p), irrev_exp_data,p0=ig,bounds=(lb,ub))

In [ ]:
b1_fit = round(popt1[0],2)
b2_fit = round(popt1[1],2)
b3_fit = round(popt1[2],2)
b4_fit = round(popt1[3],2)
# print output
print(f"b1 fit:{b1_fit}, b2 fit:{b2_fit}, b3 fit:{b3_fit}, b4 fit:{b4_fit}")

In [ ]:
cells = [3,6,9,12,15,18]
sno = 11
sim_des = f'cond{sno}'
irrev_exp_data=[]
del_sei=[]; del_li=[]; es_ic_n=[]; es_ic_p=[];
for cell in cells:
    # load the data for each cell
    # get parameters to intialize the model
    cell_no,dfe,dfe_0,dfo_0,N,N_0 = load_data(cell,eSOH_DIR,oCV_DIR)
    if cell == 3:
        Ns = np.insert(N_0[1:]-1,0,0)
        dfe = dfe_0
        Ncyc = N_0[-3]
        Ns = Ns[:-2]
    elif cell == 9 or cell == 15:
        Ns = np.insert(N[1:]-1,0,0)
        Ncyc = N[-1]
    elif cell == 12:
        Ns = np.insert(N[1:]-1,0,0)
        Ncyc = N[-1]
    else:
        Ns = np.insert(N_0[1:]-1,0,0)
        dfe = dfe_0
        Ncyc = N_0[-1]
    eps_n_data,eps_p_data,c_rate_c,c_rate_d,dis_set,Temp,SOC_0 = init_exp(cell_no,dfe,spm,parameter_values)
    experiment = pybamm.Experiment(
        [
            ("Discharge at "+c_rate_d+dis_set,
            "Rest for 10 sec",
            "Charge at "+c_rate_c+" until 4.2V", 
            "Hold at 4.2V until C/100")
        ] *Ncyc,
        termination="50% capacity",
    #     cccv_handling="ode",
    )
   
    par_val = {}
    # Room temp
    par_val[0] = [4.0312e-08,1.8157e-07,1.0776,2.3586e-09,-4.9170e-09,-1.4406e-09,4.60788219e-16,4.56607447e-19]
    # First Tuning
    par_val[1] = [5.6076e-08,6.7429e-07,1.02,1.4576e-08,-1.7447e-07,-2.5257e-08,4.60788219e-16,4.56607447e-19]
    # Contraint 1/2 and 2
    par_val[2] = [8.0624e-08,3.6314e-07,1.02,4.7172e-09,-9.8340e-09,-2.8812e-09,4.60788219e-16,4.56607447e-19]
    # Constraint 1/5 and 5
    par_val[3] = [9.8220e-08,9.0785e-07,1.02,1.1793e-08,-2.4585e-08,-7.2030e-09,4.60788219e-16,4.56607447e-19]
    # Constraint 1/5 and 5 with no constraints on beta2'
    par_val[4] = [6.2621e-08,6.8771e-07,1.02,1.1793e-08,-1.7385e-07,-2.4901e-08,4.60788219e-16,4.56607447e-19]
    # Constraint 1/2 and 2 with no constraints on beta2'
    par_val[5] = [8.0624e-08,3.6314e-07,1.002,4.7172e-09,-2.4217e-07,-2.5212e-08,4.60788219e-16,4.56607447e-19]
    # kpl=0, retune ksei
    par_val[6] = [1.2796e-07,7.2624e-07,1.02,0,-1.7764e-07,-1.4406e-11,4.9166e-17,4.56607447e-19]
    # kpl=0, retune ksei
    par_val[7] = [1.1375e-07,6.4051e-07,1.0,0,-1.5557e-07,-2.2287e-08,2.2675e-15,4.56607447e-19]
    # kpl=0, retune ksei, different cost function, better initial guess    
    par_val[8] = [9.1084e-08,5.9270e-07,1.00,0,-1.6967e-07,-3.0447e-08,1.5803e-15,4.56607447e-19]
    # kpl=0, retune ksei, cells 6 9 12
    par_val[9] = [2.8391e-07,2.0958e-06,1.9123,0,-8.6507e-07,-9.9184e-08,2.8281e-15,4.56607447e-19]
    # kpl=0, retune ksei, cells 3 9 12
    par_val[10] = [1.3292e-07,9.2142e-07,1.0,0,-1.2058e-07,-2.1779e-08,1.9308e-15,4.56607447e-19]
    # 2 step method with ksei
    par_val[11] = [1.1759e-07,8.9155e-07,1.0,7.5992e-09,-1.2611e-07,-2.3971e-08,1.4840e-15,4.56607447e-19]
    # 2 step method with dsei
    par_val[12] = [1.2057e-07,8.9269e-07,1.0,1.1355e-08,-1.3511e-07,-3.1561e-08,4.6079e-16,3.6445e-18]
    # 1 step method with dsei
    par_val[13] = [1.4653e-07,9.3369e-07,1.0,4.2122e-09,-1.3385e-07,-2.7939e-08,4.6079e-16,4.7510e-19]
    # First Tuning 3,9,12
    par_val[14] = [1.1944e-07,8.9283e-07,1.0,1.1646e-08,-1.3499e-07,-3.0872e-08,4.60788219e-16,4.56607447e-19]
    parameter_values = get_parameter_values()
    parameter_values.update(
        {
            "Negative electrode active material volume fraction": eps_n_data,
            "Positive electrode active material volume fraction": eps_p_data,
            "Initial temperature [K]": 273.15+Temp,
            "Ambient temperature [K]": 273.15+Temp,
            "Positive electrode LAM constant proportional term [s-1]": par_val[sno][0],
            "Negative electrode LAM constant proportional term [s-1]": par_val[sno][1],
            "Positive electrode LAM constant proportional term 2 [s-1]": par_val[sno][5],
            "Negative electrode LAM constant proportional term 2 [s-1]": par_val[sno][4],
            "Positive electrode LAM constant exponential term": par_val[sno][2],
            "Negative electrode LAM constant exponential term": par_val[sno][2],
            "SEI kinetic rate constant [m.s-1]":  par_val[sno][6], #1.08494281e-16 , 
            "EC diffusivity [m2.s-1]": par_val[sno][7],#8.30909086e-19,
            "SEI growth activation energy [J.mol-1]": 1.87422275e+04,#1.58777981e+04,
            "Lithium plating kinetic rate constant [m.s-1]": par_val[sno][3],
            "Initial inner SEI thickness [m]": 0e-09,
            "Initial outer SEI thickness [m]": 5e-09,
            "Li plating resistivity [Ohm.m]": 5*3e4,
            "SEI resistivity [Ohm.m]": 1.25*3e4,
            "Negative electrode partial molar volume [m3.mol-1]": 7e-06,
            "Negative electrode LAM min stress [Pa]": 0,
            "Negative electrode LAM max stress [Pa]": 0,
            "Positive electrode LAM min stress [Pa]": 0,
            "Positive electrode LAM max stress [Pa]": 0,
            "Negative electrode diffusion coefficient [m2.s-1]": 8e-14,
            "Positive electrode diffusion coefficient [m2.s-1]": 8e-15,
            # "Negative electrode critical stress [Pa]": 20e+06,
            # "Positive electrode critical stress [Pa]": 40e+06,
        },
        check_already_exists=False,
    )
    # simulate the model
    df = cycle_adaptive_simulation_V2(spm, parameter_values, experiment,SOC_0, save_at_cycles=1)
    # SEI layer thickness
    del_sei = df["X-averaged SEI thickness [m]"][Ns]-df["X-averaged SEI thickness [m]"][0]
    # Plated Lithium thickness
    del_li = df["X-averaged lithium plating thickness [m]"][Ns]
    # negative electode inactive material
    es_ic_n = -df["X-averaged negative electrode active material volume fraction"][Ns]+df["X-averaged negative electrode active material volume fraction"][0]
    # positive electrode inactive material 
    es_ic_p = -df["X-averaged positive electrode active material volume fraction"][Ns]+df["X-averaged positive electrode active material volume fraction"][0]
    irrev_exp_data = dfe["irrev_exp"].to_numpy()
    irrev_exp_data = irrev_exp_data[:len(Ns)]
    Ah_Th = dfe["Ah_th"].to_numpy()
    Ah_Th = Ah_Th[:len(Ns)]
    # equation 57 in paper
    irrev_exp = (b1_fit*del_sei*1e6+b2_fit*del_li**2*1e12+b3_fit*es_ic_n+b4_fit*es_ic_p)
    fig,ax = plt.subplots(1,1,figsize=(5,4))
    ax.plot(Ah_Th,irrev_exp_data,'kx')
    ax.plot(Ah_Th,irrev_exp,'o-')
    ax.set_xlabel('Ah Throughput')
    ax.set_ylabel(r"Expansion [$\mu$m]")
    ax.set_title(f"Cell {cell_no}")